# GPTPet v2 research scratchpad

Started 2026-09-18. Purpose: collect papers, projects and software that should shape the v2 brain, which will drive the robot only through MCP tools (ai2thor-mcp in sim, gpt-pet-mcp on the real robot).

Constraints that filter everything below:
- 4-wheeled (mecanum) robot, Kinect 360 RGB-D, RPLIDAR, IMU, proximity sensors. No arms, no legs.
- Jetson Nano onboard is weak; SLAM/nav2 already run on the server. The brain will run on the server and/or cloud.
- Latency is the primary design constraint. v1 spent up to ~10 s per loop with the robot standing still.
- Voice interaction is a planned addition.
- Agent framework should be general because all actuation is behind MCP. Google ADK is the current candidate.
- Interest in a System 1 / System 2 split and in ACT-R.

Reading checklist: sections 1-10 use task-list checkboxes. Change `[ ]` to `[x]` in the cell source as you finish each item.


## 0. What v1 (gpt-pet) taught us

Loop: sensory modules -> subconscious input modules (vision LLM + vector-DB view cache, audio, proximity) -> conscious module (task + goal) -> executor (skill library, else code-writing agent -> AST check -> mock -> real) -> report result -> sleep(5).

Where the ~10 s went (all serial, one thread):
1. Vision: a gpt-4o call per iteration unless the image-similarity cache hit (Weaviate img2vec + distance threshold).
2. Conscious: gpt-4o call, plus gpt-4o-mini summarizer for the observation, plus an OutputFixingParser call on bad YAML, plus time-weighted memory retrieval (OpenAI embeddings round trip).
3. Executor: vector search for a matching skill, a 20% random validation LLM call, or a JSON agent loop of up to 5 iterations that writes Python, mock-executes, then really executes.
4. Sim-only overhead: proximity sensors emulated by trial moves and teleport-back in four directions after every step.
5. sleep(5) after each loop so the robot is still for the next frame.

Keep from v1:
- The sensory / subconscious / conscious / executor decomposition, and the goal vs task distinction.
- Skill library with validation and deletion on failure (Voyager pattern).
- "Seen before" novelty signal from image embedding similarity.
- Passageway detection from depth with visual markers for the VLM. ai2thor-mcp now does this better (numbered markers, stable ids, navmesh validation).
- Safety gating before anything touches motors (AST check -> mock -> real).
- Generative-agent memory (recency x importance x relevance + reflection).
- Rollback of the last actions after a failure.

Reconsider in v2:
- Code as the action space. With ~10 primitives on a wheeled robot it cost an agent loop per task and needed the secret-passphrase hack to prove the tool was called. Tool calls against the MCP server remove this entirely (set_nav_goal / do_move / do_rotate).
- One serial thread. Nothing overlapped; every LLM call stopped the robot.
- Six Weaviate collections (views, goals, tasks, skills, objects, memories), each needing an embedding call.
- The conscious prompt did four jobs at once (mentor, next task, goal, goal-completed flag).
- Views were cached per goal_id, so the same view was re-described whenever the goal changed.
- Prompts assumed OpenAI YAML/JSON agent formats from LangChain 0.1.

What ai2thor-mcp already gives v2 (and gpt-pet-mcp must match 1:1):
- get_current_view: objects + passages with stable ids and a numbered image (set-of-marks).
- set_nav_goal / get_nav_status / cancel_nav: nav2 NavigateToPose semantics (async goal, polling, cancel, preemption).
- list_destinations (memory of everything seen), get_map, do_move / do_rotate (preempting teleop), overhead recording.

So the v2 brain's job reduces to: decide where to go and what to do, keep deciding while the robot moves, remember, and talk.


## 1. Fast/slow architectures (the core design question)

Language-agent level (closest to what v2 needs):

- [ ] **Agents Thinking Fast and Slow: A Talker-Reasoner Architecture** (Google DeepMind, 2024). [arXiv 2410.08328](https://arxiv.org/abs/2410.08328). Talker (System 1) is always on; Reasoner (System 2) runs asynchronously and is the only writer of a structured belief state in shared memory; the Talker reads the latest belief and may act on a stale one; it waits only in phases that need planning or tools. Failure modes: snap-judgment Talker, stale beliefs. Open question: deciding automatically when System 2 is needed. Template for v2's S1/S2 contract.
- [x] **Thinking Fast and Slow in AI** (Booch, Rossi et al., AAAI 2021). [arXiv 2010.06002](https://arxiv.org/abs/2010.06002). The original proposal; introduces the metacognitive arbiter idea.
- [x] **SOFAI project site** (IBM). [sites.google.com/view/sofai](https://sites.google.com/view/sofai/home). Overview of the Slow and Fast AI architecture and its papers.
- [x] **Thinking Fast and Slow in Human and Machine Intelligence** (CACM 2025). [ACM](https://dl.acm.org/doi/10.1145/3715709). Current statement of the SOFAI position. Key idea: a metacognitive module arbitrates between S1 (experience-based) and S2 (deliberative) solvers using confidence and cost. The arbitration policy is the real design problem.
- [ ] **A Neurosymbolic Fast and Slow Architecture for Graph Coloring** (SOFAI-v2, 2024). [arXiv 2412.01752](https://arxiv.org/html/2412.01752). Concrete arbitration mechanics in the newest SOFAI version.
- [ ] **Combining Fast and Slow Thinking for Human-like and Efficient Navigation in Constrained Environments** (2022). [arXiv 2201.07050](https://arxiv.org/pdf/2201.07050). SOFAI applied to navigation on a grid; the closest SOFAI paper to a moving robot.
- [ ] **Hi Robot: Open-Ended Instruction Following with Hierarchical Vision-Language-Action Models** (Physical Intelligence, ICML 2025). [arXiv 2502.19417](https://arxiv.org/abs/2502.19417). High-level VLM reasoner emits language subgoals and absorbs situated feedback ("that's not trash"); a low-level policy executes. Same S2-emits-subgoal, S1-executes shape, with language as the interface.

Runtime and systems papers on planning while acting:

- [ ] **TypeGo: An OS Runtime for Embodied Agents** (Jul 2026). [arXiv 2607.05482](https://arxiv.org/abs/2607.05482). Asynchronous LLM loops at multiple timescales overlapping execution; a "Skill Kernel" arbitrates typed physical subsystems among per-task processes with preempt/resume/replace; speculative skill streaming hides LLM latency; a fast first-action path gives visible feedback within a second. Unitree Go2 prototype: 50% lower per-step delay, 73% better time-to-first-action.
- [ ] **AgenticCache: Cache-Driven Asynchronous Planning for Embodied AI Agents** (Apr 2026). [arXiv 2604.24039](https://arxiv.org/abs/2604.24039). Exploits plan locality: a cache of plan transitions is served immediately while a background updater validates with the LLM. Reported 65% lower latency, 50% fewer tokens.
- [ ] **Parallelized Planning-Acting for Efficient LLM-based Multi-Agent Systems** (AAMAS 2026). [arXiv 2503.03505](https://arxiv.org/abs/2503.03505). Dual-thread planning and acting with interruptible execution.
- [ ] **Agentic AI for Robot Control: Flexible but still Fragile** (Feb 2026). [arXiv 2602.13081](https://arxiv.org/abs/2602.13081). Real deployments on two robots; explicit event checks and operator interventions that redirect execution; notes that agent frameworks lack hard preemption of tool calls, so robot skills must be cancellable and monitored concurrently. Candid about nondeterminism and prompt sensitivity.
- [ ] **Robotouille: An Asynchronous Planning Benchmark for LLM Agents** (ICLR 2025). [alphaXiv 2502.05227](https://www.alphaxiv.org/abs/2502.05227v1). Benchmark for agents that must plan over overlapping actions and interruptions.

VLA-level dual systems (manipulation-focused, but the pattern transfers):

- [ ] **Helix: A Vision-Language-Action Model for Generalist Humanoid Control** (Figure). [figure.ai/news/helix](https://www.figure.ai/news/helix). S2 VLM at 7-9 Hz, S1 policy at 200 Hz; Helix 02 added a System 0.
- [ ] **GR00T N1: An Open Foundation Model for Generalist Humanoid Robots** (NVIDIA, 2025). [arXiv 2503.14734](https://arxiv.org/html/2503.14734v1). S2 VLM at 10 Hz, S1 diffusion transformer at 120 Hz.
- [ ] **Hume: Introducing System-2 Thinking in Vision-Language-Action Model** (2025). [arXiv 2505.21432](https://arxiv.org/html/2505.21432v1).
- [ ] **A Dual Process VLA: Efficient Robotic Manipulation Leveraging VLM** (2024). [arXiv 2410.15549](https://arxiv.org/pdf/2410.15549).
- [ ] **Real-Time Execution of Action Chunking Flow Policies** (Physical Intelligence). [paper](https://www.pi.website/download/real_time_chunking.pdf). Decouple prediction from execution: keep executing the current chunk while the next is computed, and blend at the seam.
- [ ] **LeRobot docs: Asynchronous Inference**. [huggingface.co/docs/lerobot/async](https://huggingface.co/docs/lerobot/async). Practical write-up of the decoupled policy-server pattern.
- [ ] **LeRobot docs: Real-Time Chunking**. [huggingface.co/docs/lerobot/rtc](https://huggingface.co/docs/lerobot/rtc). GPTPet analogue: keep driving the current nav goal while S2 chooses the next; the "chunk" is a nav goal plus a fallback behavior.

Classic layered control (still the right vocabulary):

- [ ] **Subsumption architecture** (Brooks, 1986). [Wikipedia](https://en.wikipedia.org/wiki/Subsumption_architecture).
- [ ] **Three-layer architecture** (Gat / Firby). [Wikipedia](https://en.wikipedia.org/wiki/Three-layer_architecture). Deliberator, sequencer, reactive skills. Map: S2 = deliberator, S1 = sequencer, nav2 = skills.
- [ ] **BehaviorTree.CPP docs** (the BT engine nav2 uses). [behaviortree.dev](https://www.behaviortree.dev/). Behavior trees as the sequencer layer.
- [ ] **BTGenBot: Behavior Tree Generation for Robotic Tasks with Lightweight LLMs** (IROS 2024). [arXiv 2403.12761](https://arxiv.org/abs/2403.12761), [code](https://github.com/AIRLab-POLIMI/BTGenBot). Small LLMs emit BT XML executed by BehaviorTree.CPP with nav2 MoveTo nodes.
- [ ] **BTGenBot-2: Efficient Behavior Tree Generation with Small Language Models** (2026). [arXiv 2602.01870](https://arxiv.org/html/2602.01870).
- [ ] **LLM-as-BT-Planner** (2024). [arXiv 2409.10444](https://arxiv.org/abs/2409.10444).
- [ ] **Code-BT: A Code-Driven Approach to Behavior Tree Generation for Robot Tasks** (IJCAI 2025). [paper](https://www.ijcai.org/proceedings/2025/0980.pdf).

Working notes for the v2 design (not readings; to be validated):

- S1: an always-on loop at the rate of get_current_view. Picks among current destinations with cheap scoring (novelty, drives, last heard command), reacts to nav failure / proximity / speech, and never waits on S2. Candidate implementations: no LLM (rules + scores), a small local model, or a Flash-class model with a strict latency budget.
- S2: background reasoner that writes a belief + intent document (goal, plan, interesting things, people) to shared memory. Triggered by events: unseen view, person or animal detected, speech, nav failure, goal completed, idle timer. Never blocks S1; S1 reads the latest belief (Talker-Reasoner).
- Arbitration: explicit, inspectable, and measured (SOFAI metacognition). Preemption goes through cancel_nav.


## 2. Cognitive architectures: CoALA, ACT-R, Soar, Global Workspace Theory

Frameworks and vocabulary:

- [ ] **Cognitive Architectures for Language Agents (CoALA)** (2023). [arXiv 2309.02427](https://arxiv.org/abs/2309.02427). Working vs long-term memory (episodic, semantic, procedural), internal vs external actions, a decision loop with planning and execution. Use its vocabulary to write the v2 design doc.
- [ ] **awesome-language-agents** (companion list to CoALA). [GitHub](https://github.com/ysymyth/awesome-language-agents).
- [ ] **A Review of 40 Years of Cognitive Architecture Research** (2016). [arXiv 1610.08602](https://arxiv.org/pdf/1610.08602). Survey of the field.

ACT-R:

- [ ] **ACT-R official software and tutorials** (CMU; Lisp, with Docker + Jupyter). [act-r.psy.cmu.edu/software](https://act-r.psy.cmu.edu/software/). Do the first tutorial units to learn buffers, productions, and activation.
- [ ] **pyactr** (Python ACT-R). [GitHub](https://github.com/jakdot/pyactr).
- [ ] **python_actr** (Carleton). [GitHub](https://github.com/CarletonCognitiveModelingLab/python_actr).
- [ ] **gactar** (one model file, several ACT-R implementations). [GitHub](https://github.com/asmaloney/gactar).
- [ ] **ACT-R/E: An Embodied Cognitive Architecture for Human-Robot Interaction** (Trafton et al., NRL, JHRI 2013). [ACM](https://dl.acm.org/doi/10.5898/JHRI.2.1.Trafton). ACT-R on robots; adds configural (navigation) and manipulative space modules.
- [ ] **Human-Like Remembering and Forgetting in LLM Agents: An ACT-R-Inspired Memory Architecture** (HAI 2025). [ACM](https://dl.acm.org/doi/10.1145/3765766.3765803). ACT-R activation as the retrieval and forgetting rule for an LLM agent.
- [ ] **Cognitive LLMs: Integrating Cognitive Architectures and LLMs (LLM-ACTR)** (2024). [arXiv 2408.09176](https://arxiv.org/pdf/2408.09176).
- [ ] **Synergistic Integration of LLMs and Cognitive Architectures for Robust AI** (AAAI Symposium 2024). [AAAI](https://ojs.aaai.org/index.php/AAAI-SS/article/view/27706).
- [ ] **FadeMem: Biologically-Inspired Forgetting for Efficient Agent Memory** (2026). [arXiv 2601.18642](https://arxiv.org/html/2601.18642). Adaptive decay by relevance, access frequency, and time.
- [ ] **Human-Inspired Memory Architecture for LLM Agents** (2026). [arXiv 2605.08538](https://arxiv.org/abs/2605.08538). Consolidation, interference-based forgetting, reconsolidation on retrieval.
- [ ] **From Human Memory to AI Memory: A Survey on Memory Mechanisms in the Era of LLMs** (2025). [arXiv 2504.15965](https://arxiv.org/html/2504.15965v2).
- [ ] **NL2GenSym: Natural Language to Generative Symbolic Rules for Soar via LLMs** (2025). [arXiv 2510.09355](https://arxiv.org/pdf/2510.09355). The Soar-side equivalent.

Global Workspace Theory (what v1's conscious/subconscious naming was reaching for):

- [ ] **LIDA: A Computational Model of Global Workspace Theory and Developmental Learning** (Franklin et al.). [AAAI](https://aaai.org/papers/0011-fs07-01-011-%EF%80%A0lida-a-computational-model-of-global-workspace-theory-and-developmental-learning/), [Wikipedia](https://en.wikipedia.org/wiki/LIDA_(cognitive_architecture)).
- [ ] **CTM-AI: A Blueprint for General AI Inspired by a Model of Consciousness** (2026). [arXiv 2605.04097](https://arxiv.org/pdf/2605.04097). First practical instantiation of Blum's Conscious Turing Machine.
- [ ] **"Theater of Mind" for LLMs: A Cognitive Architecture Based on Global Workspace Theory** (Apr 2026). [arXiv 2604.08206](https://arxiv.org/abs/2604.08206). Event-driven broadcast hub, specialist agents compete for the workspace, entropy-based intrinsic drive.
- [ ] **Unified Mind Model: Reimagining Autonomous Agents in the LLM Era** (2025). [arXiv 2503.03459](https://arxiv.org/abs/2503.03459). GWT as macro-architecture; MindOS engine.

Drives and affect for a pet:

- [ ] **Kismet's homeostatic drives** (Breazeal, Designing Sociable Robots, MIT Press 2002). Needs as drives with a homeostatic band; behavior selection restores the band.
- [ ] **Emotion in Reinforcement Learning Agents and Robots: A Survey** (2017). [arXiv 1705.05172](https://arxiv.org/pdf/1705.05172).
- [ ] **Human-Robot Mutual Learning through Affective-Linguistic Interaction and Differential Outcomes Training** (2024). [arXiv 2407.01280](https://arxiv.org/pdf/2407.01280). Reachy with homeostatically controlled needs driving its motivational state.
- [ ] **Design and Implementation of a Companion Robot with LLM-Based Hierarchical Emotion Motion Generation** (Applied Sciences 2025). [DOI](https://doi.org/10.3390/app152312759).
- [ ] **A motivational-based learning model for mobile robots** (2024). [ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S138904172400072X).

Notes (not readings):

- What ACT-R can contribute to v2: (a) declarative memory activation = base-level learning (recency and frequency) + spreading activation + partial matching, a principled retrieval and forgetting rule instead of raw cosine similarity; (b) productions with utility learning as a cheap S1 policy; (c) modules and buffers as a bounded working memory; (d) explicit timing.
- GWT gives a principled answer to "what wakes System 2": modules compete to broadcast, and a salient broadcast (novel object, heard command, stuck) recruits deliberation.
- Idea: needs (energy, social contact, curiosity, play) as homeostatic variables that bias S1 scoring and S2 goals.


## 3. Memory for an embodied pet

- [ ] **Generative Agents: Interactive Simulacra of Human Behavior** (Park et al., 2023). [arXiv 2304.03442](https://arxiv.org/abs/2304.03442). v1 used LangChain's GenerativeAgentMemory; keep the recency x importance x relevance retrieval and reflection, drop the langchain_experimental dependency.
- [ ] **ReMEmbR: Building and Reasoning Over Long-Horizon Spatio-Temporal Memory for Robot Navigation** (NVIDIA, 2024). [arXiv 2409.13682](https://arxiv.org/pdf/2409.13682). VLM captions of short video segments with timestamp and pose go into a vector DB; an LLM agent queries iteratively; deployed on a wheeled Nova Carter on Jetson. Very close to "pet memory of where things and people are".
- [ ] **ReMEmbR code**. [GitHub](https://github.com/NVIDIA-AI-IOT/remembr). Read the memory builder and the query agent.
- [ ] **ReMEmbR NVIDIA technical blog**. [blog](https://developer.nvidia.com/blog/using-generative-ai-to-enable-robots-to-reason-and-act-with-remembr/).
- [ ] **Embodied-RAG: General Non-parametric Embodied Memory for Retrieval and Generation** (2024). [arXiv 2409.18313](https://arxiv.org/html/2409.18313v4). Hierarchical semantic-forest memory over robot experience for navigation and QA.
- [ ] **ConceptGraphs: Open-Vocabulary 3D Scene Graphs for Perception and Planning** (2023). [arXiv 2309.16650](https://arxiv.org/abs/2309.16650). Open-vocab 3D scene graph from posed RGB-D.
- [ ] **Clio: Real-time Task-Driven Open-Set 3D Scene Graphs** (MIT SPARK, 2024). [arXiv 2404.13696](https://arxiv.org/abs/2404.13696), [code](https://github.com/MIT-SPARK/Clio). Information-bottleneck clustering keeps only task-relevant objects and places.
- [ ] **Hydra: real-time 3D scene graph construction** (MIT SPARK). [GitHub](https://github.com/MIT-SPARK/Hydra). The places layer Clio builds on.
- [ ] **EmbodiedLGR: Lightweight Graph Representation and Retrieval for Semantic-Spatial Memory** (2026). [arXiv 2604.18271](https://arxiv.org/html/2604.18271v1).
- [ ] **A Scene Graph Backed Approach to Open Set Semantic Mapping** (2026). [arXiv 2602.03781](https://arxiv.org/pdf/2602.03781).
- [ ] **LLM-Empowered Embodied Agent for Memory-Augmented Task Planning in Household Robotics** (2025). [arXiv 2504.21716](https://arxiv.org/abs/2504.21716).
- [ ] **Mem0 vs Letta (MemGPT): AI Agent Memory Compared** (2026). [vectorize.io](https://vectorize.io/articles/mem0-vs-letta). Drop-in memory layer vs agent runtime with core / archival tiers.
- [ ] **Best AI Agent Memory Frameworks in 2026**. [atlan.com](https://atlan.com/know/best-ai-agent-memory-frameworks-2026/).

Notes (not readings):

- With Kinect RGB-D and nav2 poses a scene graph is feasible on the server GPU.
- v1's six Weaviate collections plus an embedding call each are part of the latency; a single store with ACT-R-style activation on top is worth prototyping.


## 4. Perception and navigation for a camera-only wheeled robot

Marker prompting (ai2thor-mcp's get_current_view already implements this):

- [ ] **Set-of-Mark Prompting Unleashes Extraordinary Visual Grounding in GPT-4V** (2023). [arXiv 2310.11441](https://arxiv.org/abs/2310.11441).
- [ ] **PIVOT: Iterative Visual Prompting Elicits Actionable Knowledge for VLMs** (Google, 2024). [arXiv 2402.07872](https://arxiv.org/abs/2402.07872). Read for the iterative refinement of numbered candidates.
- [ ] **VLMnav: End-to-End Navigation with Vision Language Models: Transforming Spatial Reasoning into Question-Answering** (2024). [arXiv 2411.05755](https://arxiv.org/abs/2411.05755). Read for prompt design and the ablations.

Zero-shot exploration and object-goal navigation with LLM/VLM-scored frontiers:

- [ ] **VLFM: Vision-Language Frontier Maps for Zero-Shot Semantic Navigation** (2023). [arXiv 2312.03275](https://www.emergentmind.com/papers/2312.03275). Occupancy frontiers scored by a language-grounded value map.
- [ ] **UniGoal: Towards Universal Zero-shot Goal-oriented Navigation** (CVPR 2025). [paper](https://openaccess.thecvf.com/content/CVPR2025/papers/Yin_UniGoal_Towards_Universal_Zero-shot_Goal-oriented_Navigation_CVPR_2025_paper.pdf).
- [ ] **WMNav: Integrating VLMs into World Models for Object Goal Navigation** (2025). [arXiv 2503.02247](https://arxiv.org/pdf/2503.02247).
- [ ] **PIGEON: VLM-Driven Object Navigation via Points of Interest Selection** (2025). [arXiv 2511.13207](https://arxiv.org/pdf/2511.13207).
- [ ] **DyNaVLM: Zero-Shot VLN with Dynamic Viewpoints and Self-Refining Graph Memory** (2025). [arXiv 2506.15096](https://arxiv.org/html/2506.15096v1).
- [ ] **Think, Remember, Navigate: Zero-Shot Object-Goal Navigation with VLM-Powered Reasoning** (2025). [arXiv 2511.08942](https://arxiv.org/html/2511.08942v1).
- [ ] **History-Augmented VLMs for Frontier-Based Zero-Shot Object Navigation** (2025). [arXiv 2506.16623](https://arxiv.org/pdf/2506.16623).
- [ ] **Stairway to Success: Floor-Aware Zero-Shot Object-Goal Navigation via LLM-Driven Coarse-to-Fine Exploration** (2025). [arXiv 2505.23019](https://arxiv.org/pdf/2505.23019).
- [ ] **LocalNav: Distilling Frontier VLMs and Embodied RL for On-Device Object Goal Navigation** (2026). [arXiv 2606.27871](https://arxiv.org/pdf/2606.27871).

Streaming perception (an S1 that never stops looking):

- [ ] **StreamingVLM: Real-Time Understanding for Infinite Video Streams** (NeurIPS 2025). [arXiv 2510.09608](https://arxiv.org/pdf/2510.09608).
- [ ] **LiveVLM: Efficient Online Video Understanding via Streaming-Oriented KV Cache and Retrieval** (2025). [arXiv 2505.15269](https://arxiv.org/html/2505.15269v2).
- [ ] **Event-VStream: Event-Driven Real-Time Understanding for Long Video Streams** (2026). [arXiv 2601.15655](https://arxiv.org/pdf/2601.15655).
- [ ] **Awesome-Streaming-Video-Understanding** (curated list). [GitHub](https://github.com/sotayang/Awesome-Streaming-Video-Understanding).

Hosted embodied reasoning:

- [ ] **Gemini Robotics ER 2 developer docs**. [ai.google.dev](https://ai.google.dev/gemini-api/docs/robotics-overview). Two endpoints: gemini-robotics-er-2-preview and gemini-robotics-er-2-streaming-preview (Live API, low-latency function calling). Pointing, tracking, boxes, trajectories; task orchestration against custom robot APIs; video moment finding and progress classification; thinking level trades latency.
- [ ] **Gemini Robotics ER 2 announcement** (DeepMind). [deepmind.google](https://deepmind.google/models/gemini-robotics/gemini-robotics-er/).

Edge VLMs, if any perception moves onboard:

- [ ] **Moondream** (0.5B / 2B; runs on Jetson). [moondream.ai](https://moondream.ai/).
- [ ] **SmolVLM** (Hugging Face blog). [blog](https://huggingface.co/blog/smolvlm).
- [ ] **NVIDIA live-vlm-webui: list of VLMs that run on Jetson**. [GitHub](https://github.com/NVIDIA-AI-IOT/live-vlm-webui/blob/main/docs/usage/list-of-vlms.md).
- [ ] **Getting Started with Edge AI on NVIDIA Jetson: LLMs, VLMs, and Foundation Models for Robotics** (NVIDIA blog, 2026). [blog](https://developer.nvidia.com/blog/getting-started-with-edge-ai-on-nvidia-jetson-llms-vlms-and-foundation-models-for-robotics/).
- [ ] **Getting Started with VLM on Jetson Nano** (LearnOpenCV). [article](https://learnopencv.com/vlm-on-jetson-nano/).
- [ ] **A Survey on Efficient Vision-Language Models** (2025). [arXiv 2504.09724](https://arxiv.org/pdf/2504.09724).

Notes (not readings):

- For a pet, "explore" is frontier selection scored by interestingness rather than by a target object.
- Gemma 3 4B QAT fits an Orin Nano. The original Jetson Nano is too small for any of these; the server GPU or an Orin Nano Super is the realistic host.


## 5. Skills, code as action, and curiosity

Skill libraries:

- [ ] **Voyager: An Open-Ended Embodied Agent with Large Language Models** (2023). [arXiv 2305.16291](https://arxiv.org/abs/2305.16291). v1's blueprint: automatic curriculum, skill library, iterative prompting with environment feedback.
- [ ] **ASPIRE: Agentic Skills Discovery for Robotics** (2026). [arXiv 2607.00272](https://arxiv.org/pdf/2607.00272).
- [ ] **Growing with Your Embodied Agent: A Human-in-the-Loop Lifelong Code Generation Framework** (2025). [arXiv 2509.18597](https://arxiv.org/pdf/2509.18597).
- [ ] **Efficient Skill Grounding via Code Refactoring with Small Language Models** (2026). [arXiv 2606.07999](https://arxiv.org/pdf/2606.07999).

Classics:

- [ ] **Code as Policies: Language Model Programs for Embodied Control** (2022). [arXiv 2209.07753](https://arxiv.org/abs/2209.07753).
- [ ] **Inner Monologue: Embodied Reasoning through Planning with Language Models** (2022). [arXiv 2207.05608](https://arxiv.org/abs/2207.05608).
- [ ] **Do As I Can, Not As I Say (SayCan)** (2022). [arXiv 2204.01691](https://arxiv.org/abs/2204.01691).

Code-writing agents (only if v2 keeps any code as action):

- [ ] **smolagents CodeAgent** (Hugging Face). [repo](https://github.com/huggingface/smolagents), [docs](https://huggingface.co/docs/smolagents/en/index). Agent writes Python that calls tools; MCP via ToolCollection.from_mcp with structured outputs. Replaces v1's environment_tool loop outright.

Curiosity:

- [ ] **OMNI: Open-endedness via Models of human Notions of Interestingness** (2023; evaluated in AI2-THOR). [arXiv 2306.01711](https://arxiv.org/html/2306.01711v3). Foundation model as a model of interestingness picks learnable and interesting next tasks.
- [ ] **OMNI-EPIC: Environments Programmed in Code** (2024). [arXiv 2405.15568](https://arxiv.org/abs/2405.15568).
- [ ] **Intrinsically-Motivated Humans and Agents in Open-World Exploration** (2025). [arXiv 2503.23631](https://arxiv.org/pdf/2503.23631).
- [ ] **Interestingness as an Inductive Heuristic for Future Compression Progress** (2026). [arXiv 2605.14831](https://arxiv.org/pdf/2605.14831).

Notes (not readings):

- Default plan is plain tool calls against the MCP server instead of generated code.
- The interestingness model is the pet's curiosity; it can score frontiers and destinations in S2 and cache scores for S1.


## 6. Agent framework and MCP plumbing

Google ADK:

- [ ] **ADK site**. [adk.dev](https://adk.dev/).
- [ ] **ADK MCP tools page**. [adk.dev/tools-custom/mcp-tools](https://adk.dev/tools-custom/mcp-tools/). McpToolset with StdioConnectionParams, SseConnectionParams, StreamableHTTPConnectionParams (ai2thor-mcp already speaks streamable HTTP on /mcp), tool_filter, asyncio lifecycle caveats.
- [ ] **ADK bidi streaming dev guide, part 1**. [adk.dev/streaming/dev-guide/part1](https://adk.dev/streaming/dev-guide/part1/).
- [ ] **ADK bidi streaming dev guide, part 5: audio, images, video**. [guide](https://google.github.io/adk-docs/streaming/dev-guide/part5/).
- [ ] **Codelab: Introduction to ADK Gemini Live API Toolkit**. [codelab](https://codelabs.developers.google.com/intro-to-adk-live).
- [ ] **ADK release notes** (v2.9.0, 2026-09-10: MCP SDK 2.x support, LiveKit runner for voice, YAML graph workflows, model failover). [GitHub releases](https://github.com/google/adk-python/releases).

Framework comparisons:

- [ ] **Comparing Open-Source AI Agent Frameworks** (Langfuse). [blog](https://langfuse.com/blog/2025-03-19-ai-agent-comparison).
- [ ] **AI Agent Frameworks (2026 Update): 8 SDKs Compared** (morphllm). [article](https://www.morphllm.com/ai-agent-framework).
- [ ] **AI Agent Frameworks Compared (2026)** (Agentmelt). [article](https://agentmelt.com/blog/ai-agent-frameworks-compared-2026/).

ROS 2 MCP servers (reference for the real-robot server):

- [ ] **robotmcp/ros-mcp-server** (rosbridge-based; topics, services, actions; ROS 1 and 2). [GitHub](https://github.com/robotmcp/ros-mcp-server).
- [ ] **Yutarop/ros-mcp**. [GitHub](https://github.com/Yutarop/ros-mcp).
- [ ] **kakimochi/ros2-mcp-server** (FastMCP node publishing cmd_vel). [GitHub](https://github.com/kakimochi/ros2-mcp-server).
- [ ] **sandraschi/ros-mcp** (full ROS 2 surface: topics, services, params, bags, launch). [GitHub](https://github.com/sandraschi/ros-mcp).
- [ ] **wise-vision/ros2_mcp**. [GitHub](https://github.com/wise-vision/ros2_mcp).
- [ ] **lpigeon/ros_mcp_server**. [GitHub](https://github.com/lpigeon/ros_mcp_server).

NVIDIA reference for a wheeled Jetson robot with an LLM agent:

- [ ] **ReMEmbR on Nova Carter** (The Robot Report). [article](https://www.therobotreport.com/remembr-generative-ai-enables-robots-reason-act-says-nvidia/).
- [ ] **Isaac ROS**. [developer.nvidia.com/isaac](https://developer.nvidia.com/isaac).

Notes (not readings):

- To check: whether a live (bidi) ADK agent can call MCP tools with the same lifecycle as an LlmAgent, and how long-running tools (nav goals) surface as background work.
- Since tools are MCP, the framework choice is really about async and streaming primitives, interrupt and cancel semantics, and voice support. Alternatives: LangGraph (checkpointing, interrupts), Pydantic AI, OpenAI Agents SDK, Claude Agent SDK, Strands, Microsoft Agent Framework, smolagents.
- Position: do not expose raw ROS to the brain; keep the ai2thor-mcp contract and implement it over rclpy nav2 actions. The ROS MCP repos are useful for transport and lifecycle patterns only.


## 7. Talking to the robot

Hosted and framework options:

- [ ] **Gemini Live API docs**. [ai.google.dev](https://ai.google.dev/gemini-api/docs/live-api). ADK bidi streaming sits on it.
- [ ] **Pipecat: Building with Gemini Live**. [docs](https://docs.pipecat.ai/pipecat/features/gemini-live).
- [ ] **Voice AI Agents on LiveKit: 2026 Engineer's Playbook**. [article](https://www.forasoft.com/blog/article/voice-ai-agents-livekit-guide). Turn detection, noise cancellation; ADK 2.9 ships a LiveKit runner.
- [ ] **Pipecat vs LiveKit Agents**. [article](https://www.dograh.com/feeds/blog/pipecat-livekit-agents).
- [ ] **Voice AI agents in production 2026: LiveKit Agents, OpenAI Realtime, Pipecat, Vapi, Retell compared**. [article](https://www.reactify-solutions.com/articles/voice-ai-agents-production-2026).

Open and local:

- [ ] **huggingface/speech-to-speech** (repo). [GitHub](https://github.com/huggingface/speech-to-speech). Silero VAD -> Parakeet STT -> LLM -> Qwen3-TTS or Kokoro, exposed as an OpenAI Realtime-compatible /v1/realtime WebSocket.
- [ ] **Hugging Face and Cerebras bring Gemma 4 to real-time voice AI** (July 2026). [blog](https://huggingface.co/blog/cerebras-gemma4-voice-ai). The stack powering 9,000+ Reachy Minis.
- [ ] **Reachy Mini goes fully local** (Hugging Face blog). [blog](https://huggingface.co/blog/local-reachy-mini-conversation).
- [ ] **Kyutai: Moshi (full duplex, ~200 ms), Unmute, Kyutai TTS**. [kyutai.org/tts](https://kyutai.org/tts/).
- [ ] **faster-whisper + Silero VAD + Kokoro offline chatbot** (example repo). [GitHub](https://github.com/AliBinary/local-ai-voice-chatbot).

Notes (not readings):

- Speech is an S1 input event that can recruit S2; barge-in maps to cancel_nav; the reply voice should not wait on S2 (Talker-Reasoner).


## 8. Companion and pet robot reference projects

- [ ] **Reachy Mini docs** (Pollen Robotics / Hugging Face). [docs](https://huggingface.co/docs/reachy_mini/en/index). Behaviors published on the Hub, expressive idle motion, VAD -> STT -> LLM -> TTS pipeline. Best current example of an open LLM companion product.
- [ ] **Reachy Mini launch post**. [blog](https://huggingface.co/blog/reachy-mini).
- [ ] **wire-pod** (Anki Vector server). [GitHub](https://github.com/kercre123/wire-pod). Local voice server that routes unmatched utterances to an LLM; Vector's original emotion engine and behavior system are worth studying for pet idle behavior.
- [ ] **pidog-embodiment**. [GitHub](https://github.com/rockywuest/pidog-embodiment). Brain/body split over HTTP, LLM-driven autonomous behaviors, works with any LLM.
- [ ] **Ai-Robo-Dog**. [GitHub](https://github.com/marceld23/Ai-Robo-Dog).
- [ ] **Petoi OpenCat**. [GitHub](https://github.com/PetoiCamp/OpenCat-Quadruped-Robot).
- [ ] **Everything-LLMs-And-Robotics** (big index). [GitHub](https://github.com/jrin771/Everything-LLMs-And-Robotics).


## 9. Evaluation

- [ ] **EmbodiedBench: Comprehensive Benchmarking MLLMs for Vision-Driven Embodied Agents** (ICML 2025 oral). [arXiv 2502.09560](https://arxiv.org/pdf/2502.09560), [code](https://github.com/EmbodiedBench/EmbodiedBench). AI2-THOR navigation and household tasks with Gym-style APIs.
- [ ] **EmbodiedEval: Evaluate Multimodal LLMs as Embodied Agents** (2025). [arXiv 2501.11858](https://arxiv.org/pdf/2501.11858).
- [ ] **Robotouille** (asynchronous planning benchmark; listed in section 1).

Notes (not readings):

- GPTPet-specific metrics to define early: fraction of wall-clock time moving, time-to-first-action after boot, novel views per minute, latency from person detected to reaction, nav success rate, cost per hour.


## 10. Surveys

- [ ] **Large Language Models for Robotics: A Survey** (2023). [arXiv 2311.07226](https://arxiv.org/html/2311.07226v2).
- [ ] **A Survey on Robotics with Foundation Models: toward Embodied AI** (2024). [arXiv 2402.02385](https://arxiv.org/abs/2402.02385).
- [ ] **Foundation Models in Robotics: A Comprehensive Review of Methods, Models, Datasets, Challenges and Future Research Directions** (2026). [arXiv 2604.15395](https://arxiv.org/html/2604.15395v1).
- [ ] **Embodied AI: From LLMs to World Models** (2025). [arXiv 2509.20021](https://arxiv.org/pdf/2509.20021).
- [ ] **Vision-and-Language Navigation Today and Tomorrow: A Survey in the Era of Foundation Models** (2024). [arXiv 2407.07035](https://arxiv.org/pdf/2407.07035).
- [ ] **Vision-Language-Action Models: Concepts, Progress, Applications and Challenges** (2025). [arXiv 2505.04769](https://arxiv.org/pdf/2505.04769).


## 11. Suggested reading order

1. Talker-Reasoner, the SOFAI CACM article, TypeGo, Parallelized Planning-Acting. These settle the fast/slow mechanics and the arbitration question.
2. CoALA, ACT-R/E, the ACT-R-inspired memory paper. Vocabulary and the memory math.
3. ReMEmbR (read the code), PIVOT, VLMnav. Memory and perception that fit the existing MCP tools.
4. ADK MCP docs and streaming guide, Gemini Robotics ER 2 docs. Plumbing.
5. OMNI. Curiosity as the pet's motivation.
6. Reachy Mini docs and huggingface/speech-to-speech. Voice reference implementation.


## 12. Open questions to resolve

- What wakes System 2: fixed events, GWT-style competition, or SOFAI-style metacognition on S1 confidence?
- S1 model choice and latency budget: rules only, a small local model, or a Flash-class model? Measure get_current_view round trip first.
- Belief-state schema shared between S1 and S2, and who is allowed to write which fields.
- Memory store: keep Weaviate, or one store with ACT-R activation on top?
- How the real robot's MCP reports stuck and collision events with the same semantics as the simulator.
- Where the brain runs: server GPU, cloud, or split by system.
